# MindCare: Emotion Classification Pipeline
This notebook details the feature extraction, model comparison, probability calibration, and evaluation for MindCare's emotion classifier.

### Target Classes (6 Categories):
* `Angry`, `Anxious`, `Calm`, `Hopeful`, `Neutral`, `Sad`


In [ ]:
import re
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix


## 1. Data Ingestion & Preprocessing
Loads raw emotion statements and applies text normalization.


In [ ]:
DATA_PATH = "../datasets/raw/Emotion_classify_Data.csv"

df = pd.read_csv(DATA_PATH)
df = df.rename(columns={"Comment": "text", "Emotion": "label"})
df = df.dropna(subset=["text", "label"])

print(f"Loaded emotion records: {len(df)}")
df.head()


In [ ]:
def clean_text(text: str) -> str:
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+', ' ', text)
    text = re.sub(r'[^a-zA-Z\s!?,.]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['text'] = df['text'].apply(clean_text)
df = df.drop_duplicates(subset=['text'])
df = df[df['text'].str.len() > 0]

print(f"Cleaned unique records: {len(df)}")
print("\nClass Distribution:")
print(df['label'].value_counts())


## 2. Train / Test Split & TF-IDF Extraction


In [ ]:
train_df, test_df = train_test_split(
    df, 
    test_size=0.2, 
    random_state=42, 
    stratify=df['label']
)

vectorizer = TfidfVectorizer(
    max_features=20000,
    ngram_range=(1, 2),
    min_df=2,
    sublinear_tf=True
)

X_train = vectorizer.fit_transform(train_df['text'])
X_test = vectorizer.transform(test_df['text'])
y_train = train_df['label']
y_test = test_df['label']
labels = sorted(y_train.unique().tolist())

print(f"Vocabulary size: {len(vectorizer.vocabulary_)}")


## 3. Model Comparison: LinearSVC vs. CalibratedClassifierCV vs. Logistic Regression


In [ ]:
# 1. Uncalibrated LinearSVC
print("Training LinearSVC...")
svc = LinearSVC(class_weight='balanced', random_state=42)
svc.fit(X_train, y_train)

# 2. Calibrated LinearSVC (Platt Scaling)
print("Training CalibratedClassifierCV...")
calibrated_svc = CalibratedClassifierCV(svc, cv=5)
calibrated_svc.fit(X_train, y_train)

# 3. Logistic Regression
print("Training LogisticRegression...")
logreg = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
logreg.fit(X_train, y_train)


## 4. Evaluation & Metrics Comparison


In [ ]:
print("=== LinearSVC Classification Report ===")
print(classification_report(y_test, svc.predict(X_test), target_names=labels))

print("=== Calibrated LinearSVC Classification Report ===")
print(classification_report(y_test, calibrated_svc.predict(X_test), target_names=labels))

print("=== Logistic Regression Classification Report ===")
print(classification_report(y_test, logreg.predict(X_test), target_names=labels))


In [ ]:
# Confusion Matrix for Calibrated LinearSVC
cm = confusion_matrix(y_test, calibrated_svc.predict(X_test), labels=labels)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Purples', xticklabels=labels, yticklabels=labels)
plt.title("Emotion Classification: Confusion Matrix")
plt.xlabel("Predicted Emotion")
plt.ylabel("True Emotion")
plt.tight_layout()
plt.show()


## 5. Softmax Temperature Calibration Demonstration
In production, MindCare uses calibrated softmax probabilities over the decision boundary margins to ensure low-confidence noise (< 25.0%) is not hallucinated.


In [ ]:
def softmax_with_temperature(decision_scores, temperature=1.0):
    exp_scores = np.exp(decision_scores / temperature)
    return exp_scores / np.sum(exp_scores, axis=-1, keepdims=True)

sample_text = ["I am feeling so joyful and happy today!"]
sample_vec = vectorizer.transform(sample_text)
decision_margins = svc.decision_function(sample_vec)[0]
calibrated_probs = softmax_with_temperature(decision_margins, temperature=1.0)

for label, prob in zip(labels, calibrated_probs):
    print(f"{label:12s}: {prob * 100:.2f}%")


## 6. Export Demonstration


In [ ]:
import os
os.makedirs("exports", exist_ok=True)

with open("exports/demo_emotion_model.pkl", "wb") as f:
    pickle.dump(svc, f)

with open("exports/demo_emotion_vectorizer.pkl", "wb") as f:
    pickle.dump(vectorizer, f)

print("Demo models exported safely.")
